# كود التنظيف: من `legal_articles_full_updated.csv` إلى `legal_articles_full.csv`

يعمل في Google Colab على أي جهاز: **Runtime ← Run all**، ثم ارفعي ملفات المدخلات معًا عند الطلب (تحديد متعدد)، وفي النهاية ينزل `legal_articles_full.csv` — وهو مدخل `model_matching.ipynb`.

**المدخلات (من مجلد `data_raw/` في المستودع):**
- `legal_articles_full_updated.csv` — ملف المواد من مشروع الفريق (176 مادة).
- `legal_articles_errors_additions.xlsx` — الأسطر الدخيلة، و«ء» الزائدة، والأسطر الساقطة من القراءة الآلية.
- `مراجعة_المواضيع.xlsx` — المواضيع «الواضحة» المعتمدة.
- `مراجعة_القانوني_المواضيع.xlsx` و`مفتاح_مراجعة_القانوني_لا_يُرسل.csv` — قرارات المختص القانوني ومفتاحها.
- `legal topic classification.py` و`legal_text_extraction.py` — كود تصنيف المواضيع من مشروع الفريق.

كتب الكود Claude (نموذج ذكاء اصطناعي من Anthropic) بطلب من فجر؛ فجر حددت المطلوب واعتمدت القرارات.

In [ ]:
!pip -q install scikit-learn openpyxl

## رفع المدخلات

In [ ]:
import os, re, unicodedata
from google.colab import files
up = files.upload()                                   # اختاري الملفات السبعة معًا
NEED = ['legal_articles_full_updated.csv', 'legal_articles_errors_additions.xlsx', 'مراجعة_المواضيع.xlsx',
        'legal topic classification.py', 'legal_text_extraction.py']
OPT = ['مراجعة_القانوني_المواضيع.xlsx', 'مفتاح_مراجعة_القانوني_لا_يُرسل.csv']
W = '/content/clean_inputs'; os.makedirs(f'{W}/src', exist_ok=True); open(f'{W}/src/__init__.py', 'w').close()
def base(n):                                          # اسم الملف دون «(1)» التي يضيفها Colab للتكرار، وبصيغة NFC
    n = unicodedata.normalize('NFC', os.path.basename(n))
    return re.sub(r' ?\(\d+\)(?=\.[^.]+$)', '', n)
got = {}
for name, data in up.items():
    b = base(name)
    if b not in NEED + OPT: print('تُجوهل (ليس من المدخلات):', name); continue
    dst = f'{W}/src/{b}' if b == 'legal_text_extraction.py' else f'{W}/{b}'
    open(dst, 'wb').write(data); got[b] = dst
miss = [n for n in NEED if n not in got]
assert not miss, f'ناقص: {miss}'
LAW = [got[n] for n in OPT if n in got]
assert len(LAW) in (0, 2), 'ملف القانوني يحتاج مفتاحه معه'
print('المدخلات جاهزة:', len(got), '| قرارات القانوني:', 'مُضمّنة' if LAW else 'غير مُضمّنة')

## كود التنظيف

In [ ]:
# -*- coding: utf-8 -*-
"""تنظيف ملف مواد الوثيقتين (قواعد الحوكمة 47 مادة + اللائحة التنفيذية 129 مادة) في ملف واحد.
من نسخة زميلة التنظيف (legal_articles_full_updated.csv) إلى legal_articles_full.csv المنظف.

الخطوات:
  1 الإضافات: حذف نص دخيل، تصحيح «ء» زائدة، إضافة الأسطر الساقطة من القراءة الآلية (عمود text)
  2 تصحيحات أخطاء القراءة الآلية (قائمة FIX، مؤكدة بطبقة نص PDF أو صور الصفحات) وحذف أرقام البنود (text)
  3 إعادة بناء text_raw بالإملاء الأصلي (بلا تشكيل ولا علامات اتجاه خفية، وأرقام لاتينية)
  4 تصحيحات إملاء text_raw، واختبار القبول: توحيد text_raw بدالة nz يعطي text حرفيًا في كل الأسطر
  5 الحوكمة م42، أعمدة الموضوعات بكود زميلة التنظيف (legal topic classification.py)، المواضيع «الواضحة»،
    وقرارات المختص القانوني إن وُجد ملفه، واللائحة م111 تأخذ موضوع توأمها م86

التشغيل (المدخلات في مجلد «المدخلات» بجانب هذا الملف):
  python3 clean_legal_articles.py
أو بمسارات صريحة:
  python3 clean_legal_articles.py <legal_articles_full_updated.csv> <legal_articles_errors_additions.xlsx> \
      <legal topic classification.py> <مجلد مشروع زميلة التنظيف (فيه src/)> <مراجعة_المواضيع.xlsx> <مجلد الناتج> \
      [<ملف القانوني المعبأ.xlsx> <مفتاح_مراجعة_القانوني_لا_يُرسل.csv>]
الناتج: <مجلد الناتج>/legal_articles_full.csv + سجلات الخطوات.
المتطلبات: pandas، openpyxl، scikit-learn.
كتب الكود Claude (نموذج ذكاء اصطناعي من Anthropic) بطلب من فجر؛ فجر حددت المطلوب واعتمدت القرارات."""
import sys, os, io, re, difflib, collections, importlib.util
import pandas as pd, openpyxl

# ---------- قائمة تصحيحات عمود text (من fixes.py) ----------
# (doc, article, wrong, right) — applied to `text` (normalized form). wrong matched as exact substring.
FIX = [
 ('reg',5,'مهنيء','مهني'),('reg',84,'متفرغاء','متفرغا'),('reg',102,'مناطقيء','مناطقي'),('reg',116,'قروضء','قروض'),
 ('gov',39,'لهماء','لهما'),('reg',112,'اعضايهماء','اعضايهما'),('reg',115,'ميلادياء','ميلاديا'),('reg',116,'والاستثمارء','والاستثمار'),
 ('reg',3,'اساشاء','اساسا'),('reg',3,'مقا','معا'),('reg',41,'مقا','معا'),('reg',3,'لفايات','لغايات'),
 ('reg',72,'مجلش','مجلس'),('reg',72,'كياتا','كيانا'),('reg',32,'يبجوز','يجوز'),
 ('reg',14,'التشسيق','التنسيق'),('reg',53,'بالتشنسيق','بالتنسيق'),('gov',35,'حشضور','حضور'),('gov',35,'1 ديم','1 تقديم'),
 ('gov',38,'المنبيقه','المنبثقه'),('gov',24,'منبيقه','منبثقه'),('reg',23,'المرشحينن','المرشحين'),('reg',51,'المنعصوص','المنصوص'),
 ('reg',112,'واللوايج','واللوايح'),('reg',123,'الثالته','الثالثه'),('reg',56,'رييشا','رييسا'),('reg',114,'ويحدت','ويحدث'),
 ('reg',114,'خاضا','خاصا'),('reg',31,'خاضا','خاصا'),('reg',61,'خاضا','خاصا'),('gov',33,'ويحشن','ويحسن'),
 ('gov',6,'مقسوماا','مقسوما'),('gov',8,'باستفلال','باستغلال'),('reg',116,'التشفيلي','التشغيلي'),('gov',29,'التشفيل','التشغيل'),
 ('reg',95,'ادنهيت','انهيت'),('reg',5,'بحتي','بحثي'),('reg',5,'بيني','بييي'),('reg',84,'تنفيد','تنفيذ'),('reg',98,'تنفيد','تنفيذ'),
 ('reg',59,'التنفيد','التنفيذ'),('reg',79,'فغاله','فعاله'),('reg',107,'فقاله','فعاله'),('gov',43,'الموققه','الموقعه'),
 ('reg',12,'ترخيها','ترخيصا'),('reg',51,'ترخيها','ترخيصا'),('reg',42,'لفرض','لغرض'),('gov',27,'ببين','يبين'),
 ('reg',58,'ان نظمه','انظمه'),('reg',58,'االزاميه','الزاميه'),('reg',58,'اش خاص','اشخاص'),('reg',58,'0.اره','0.اداره'),
 ('reg',65,'اس تحاله','استحاله'),('reg',11,'اس تكمال','استكمال'),('reg',51,'اس تكماله','استكماله'),('reg',26,'مس بب','مسبب'),
 ('reg',62,'والمحاس بين','والمحاسبين'),('reg',87,'قرا ر','قرار'),('reg',125,'والموس سات','والموسسات'),('reg',16,'تقد','تعد'),
 ('reg',34,'موجما','موجها'),('reg',48,'دوي','ذوي'),('reg',83,'نافدا','نافذا'),('reg',98,'متفرعا','متفرغا'),
 ('reg',101,'الاني','الاتي'),('reg',101,'الجفرافي','الجغرافي'),('reg',104,'عضو ا','عضوا'),('reg',113,'ممايله','مماثله'),
 ('reg',121,'مسوعا','مسوغا'),('reg',123,'مشايه','مشابه'),('gov',4,'فيرخل','فيرحل'),('gov',7,'منج','منح'),('gov',7,'تفيب','تغيب'),
 ('gov',7,'ولا يبت بحضوره','ولا يثبت بحضوره'),('gov',9,'والفرض','والغرض'),('gov',23,'بالاني','بالاتي'),('gov',23,'واقتراحج','واقتراح'),
 ('gov',35,'بالاني','بالاتي'),('gov',32,'مبلفا','مبلغا'),('gov',40,'الاني','الاتي'),('reg',105,'حج-','ح-'),
 ('reg',22,'7/00','50%'),('reg',77,'(10/)','(25%)'),('reg',94,'(710)','(25%)'),('reg',100,'01 96','20%'),('reg',100,'للاخير 10 من','للاخير 30% من'),
 ('gov',5,'مع 5 من','مع 25% من'),('gov',6,'9049','49%'),('gov',13,'49/؟','49%'),('gov',14,'949','49%'),('gov',29,'9085','85%'),
 ('reg',22,')1(','(1)'),('reg',26,')1(','(1)'),('reg',114,')1(','(1)'),
]
for a in [65,73,76,77,93,107,109]: FIX.append(('reg',a,'الانيه','الاتيه'))
FIX.append(('gov',29,'الانيه','الاتيه'))
for k,a in [('reg',77),('reg',94),('gov',30)]: FIX.append((k,a,'للاني','للاتي'))
# gov6 formula: lines 8-9 merged into one line with the formula from the page image
FORMULA=('gov',6,'عدد الاصوات (زمضه | لداع - قيمه التبرعات غير المقيده للعضو الداعم + قيمه رسم العضويه\nقيمه رسم العضويه',
         'عدد الاصوات للعضو الداعم = (قيمه التبرعات غير المقيده للعضو الداعم + قيمه رسم العضويه) / قيمه رسم العضويه')
FIX.append(FORMULA)
FIX += [('reg',75,'تقد','تعد'),('reg',79,'الفقره (1)','الفقره (6)'),('reg',107,'الفقره (1)','الفقره (6)'),('reg',106,'( ا|)','(2)')]

# ---------- التوحيد والتنظيف (من step4.py؛ nz مطابقة لدالة زميلة التنظيف) ----------
DIAC=re.compile(r"[ؐ-ًؚ-ٰٟۖ-ۭـ]")
BIDI=re.compile(r"[‎‏‪-‮⁦-⁩﻿؜]")
DIG=str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹","01234567890123456789")
def nz(t):  # identical to colleague's nz()
    t=DIAC.sub("",t).translate(DIG); t=re.sub("[إأآٱ]","ا",t)
    t=t.replace("ى","ي").replace("ة","ه").replace("ؤ","و").replace("ئ","ي").replace("‏","").replace("‎","")
    return re.sub(r"\s+"," ",t).strip()
def clean(t): return BIDI.sub("",DIAC.sub("",t)).translate(DIG)
AR='ء-ي'
def core(tok):
    m=re.match(rf'^([^{AR}]*)([{AR}].*[{AR}]|[{AR}])([^{AR}]*)$',tok); return m.groups() if m else ('',tok,'')


# ---------- الخطوة 1 ----------
def step1_additions(src, addx, logdir):
    """نص دخيل، «ء» زائدة، والأسطر الساقطة من القراءة الآلية — على عمود text (من apply_additions.py)."""
    d = pd.read_csv(src)
    x = pd.read_excel(addx, sheet_name=None)
    DK = {'اللائحة التنفيذية': 'reg', 'قواعد الحوكمة': 'gov'}
    lines = {(r.doc_key, r.article_no): r.text.split('\n') for r in d.itertuples()}
    deleted = set(); log = []
    for _, r in x['إضافات عمود text'].iterrows():
        k = (DK[r['الوثيقة']], r['رقم المادة']); i = r['رقم السطر داخل المادة'] - 1; e = r['رقم الخطأ']
        old = str(r['النص كما في الملف']); L = lines[k]
        if e == 2:
            log.append((*k, i+1, e, old, 'لم يُطبَّق (قرار مفتوح)')); continue
        assert old in L[i], (k, i, old)
        if e == 1:
            if L[i].strip() == old.strip():
                deleted.add((k, i)); log.append((*k, i+1, e, old, 'حُذف السطر'))
            else:
                L[i] = L[i].replace(old, '').rstrip(); log.append((*k, i+1, e, old, 'حُذف من السطر'))
        elif e == 3:
            assert L[i].count(old) == 1, (k, i, old)
            L[i] = L[i].replace(old, str(r['التصحيح المقترح'])); log.append((*k, i+1, e, old, 'صُحح إلى ' + str(r['التصحيح المقترح'])))
    ins = {}
    for _, r in x['أسطر ساقطة من OCR'].iterrows():
        k = (DK[r['الوثيقة']], r['رقم المادة']); n = r['يُضاف بعد السطر رقم (في text)']
        assert lines[k][n-1].strip() == str(r['السطر الذي يُضاف بعده (في text)']).strip(), (k, n)
        ins.setdefault(k, {}).setdefault(n-1, []).append(str(r['النص بالكتابة الموحدة (لعمود text)']).strip())
        log.append((*k, n, 'ساقط', r['النص بالكتابة الموحدة (لعمود text)'], 'أُضيف سطرًا بعد السطر ' + str(n)))
    new = []
    for r in d.itertuples():
        k = (r.doc_key, r.article_no); res = []
        for i, ln in enumerate(lines[k]):
            if (k, i) not in deleted: res.append(ln)
            res += ins.get(k, {}).get(i, [])
        new.append('\n'.join(res))
    d['text'] = new
    buf = io.StringIO(); d.to_csv(buf, index=False); buf.seek(0)   # كما في السلسلة الأصلية: كتابة CSV ثم إعادة قراءته
    lg = pd.DataFrame(log, columns=['doc_key','article_no','line','error','text','action'])
    lg.to_csv(os.path.join(logdir, 'سجل_1_الإضافات.csv'), index=False, encoding='utf-8-sig')
    print('  تغيّرت مواد:', sum(a!=b for a,b in zip(pd.read_csv(src).text, d.text)))
    return pd.read_csv(buf)


# ---------- الخطوة 2 ----------
def step2_fixes_and_markers(d, logdir):
    """تصحيحات أخطاء القراءة الآلية (FIX) وحذف أرقام البنود من أول الأسطر — على عمود text (من step123.py)."""
    T={(r.doc_key,r.article_no):r.text for r in d.itertuples()}
    log=[]
    # steps 1-2: word fixes
    for k,a,w,r in FIX:
        t=T[(k,a)]; pat=re.compile(r'(?<![\u0621-\u064A0-9])'+re.escape(w)+r'(?![\u0621-\u064A0-9])')
        n=len(pat.findall(t)); assert n>=1,(k,a,w)
        if t.count(w)!=n: print('substring-only occurrences skipped:',k,a,w,t.count(w)-n)
        T[(k,a)]=pat.sub(lambda m:r,t); log.append((k,a,'تصحيح',w,r,n))
    # step 3: list markers at line start
    AR='ء-ي'
    X=rf'(?:[^\s{AR}]{{0,3}}[{AR}]?[^\s{AR}]{{0,3}}|عا|لا|حج)'
    P1=re.compile(rf'^(?:[^\s{AR}]{{1,3}}\s+|[{AR}]\s+)?{X}[.\-)/](?=\s|[{AR}(])\s*')
    P2=re.compile(rf'^\d{{1,2}}(?:\s+\d{{1,2}})?\s+(?=[{AR}])')
    P3=re.compile(r'^[©*|!"\'“”\[\]/]+\s+')
    P4=re.compile(rf'^[^\s{AR}(«]{{1,4}}\s+(?=[{AR}])')
    P5=re.compile(rf'^[“”"\'|][{AR}]\s+')
    removed=collections.Counter()
    for key,t in T.items():
        out=[]
        for i,ln in enumerate(t.split('\n'),1):
            s=ln.strip(); pre=''
            for _ in range(4):
                m=P1.match(s) or P2.match(s) or P3.match(s) or P4.match(s) or P5.match(s)
                if not m or m.end()==0: break
                pre+=m.group(); s=s[m.end():]
            mt=re.search(r'(?<=[.:؛,])\s*\d{1,2}\s*$',s)
            if mt: log.append((key[0],key[1],'رقم بند آخر السطر',mt.group().strip(),'',i)); s=s[:mt.start()]
            if pre: removed[pre.strip()]+=1; log.append((key[0],key[1],'رقم بند',pre.strip(),'',i))
            if re.fullmatch(r'[\d\W]*',s) and s!='':
                removed['سطر رقم فقط:'+s]+=1; log.append((key[0],key[1],'سطر رقم فقط',s,'',i)); continue
            out.append(s)
        T[key]='\n'.join(out)
    d['text']=[T[(r.doc_key,r.article_no)] for r in d.itertuples()]
    pd.DataFrame(log,columns=['doc','art','type','old','new','n_or_line']).to_csv(os.path.join(logdir,'سجل_2_التصحيحات_وأرقام_البنود.csv'),index=False,encoding='utf-8-sig')
    print('  أرقام بنود حُذفت:', sum(removed.values()))
    return d


# ---------- الخطوة 3 ----------
def step3_rebuild_text_raw(d, addx, logdir):
    """إعادة بناء text_raw بالإملاء الأصلي بمحاذاة الكلمات مع text المنظف (من step4.py)."""
    ADDX = addx
    n=d
    D=collections.defaultdict(collections.Counter)
    for t in n.text_raw:
        for tok in clean(t).split():
            a,c,b=core(tok)
            if re.fullmatch(rf'[{AR}]+',c): D[nz(c)][c]+=1
    ad=pd.read_excel(ADDX,sheet_name='أسطر ساقطة من OCR')
    for t in ad['النص الساقط كما في PDF (لعمود text_raw)']:
        for tok in clean(str(t)).split():
            a,c,b=core(tok)
            if re.fullmatch(rf'[{AR}]+',c): D[nz(c)][c]+=1000
    DICT_LOG=[]
    stats=collections.Counter(); fallback=collections.Counter(); out=[]
    for r in n.itertuples():
        R=clean(r.text_raw).split(); NR=[nz(x) for x in R]
        lines=r.text.split('\n'); T=[w for ln in lines for w in ln.split()]
        sm=difflib.SequenceMatcher(None,NR,T,autojunk=False); M={}; BLK={}
        for op,i1,i2,j1,j2 in sm.get_opcodes():
            if op=='equal':
                for k in range(j2-j1): M[j1+k]=R[i1+k]
            else:
                for j in range(j1,j2): BLK[j]=(i1,i2)
        res=[]
        for j,w in enumerate(T):
            if j in M: res.append(M[j]); stats['aligned']+=1; continue
            best=None
            if j in BLK:
                i1,i2=BLK[j]
                for i in range(max(0,i1-1),min(len(R),i2+1)):
                    rt=difflib.SequenceMatcher(None,NR[i],w,autojunk=False).ratio()
                    if rt>=0.6 and (best is None or rt>best[0]): best=(rt,i)
            if best:
                src=R[best[1]]; ns=nz(src)
                if len(ns)==len(src):   # 1:1 letters, so spelling can be transferred position by position
                    cm=difflib.SequenceMatcher(None,ns,w,autojunk=False); o=''
                    for op,a1,a2,b1,b2 in cm.get_opcodes():
                        o+= src[a1:a2] if op=='equal' else w[b1:b2]
                    if nz(o)==w: res.append(o); stats['transferred']+=1; continue
            a,c,b=core(w)
            if c in D or any(nz(s)==c for s in D.get(c,{})):
                sp=D[c].most_common(1)[0][0]; res.append(a+sp+b); stats['dictionary']+=1; DICT_LOG.append((r.doc_key,r.article_no,w,a+sp+b,dict(D[c])))
            else:
                res.append(w); stats['kept_normalized']+=1
                if re.search(rf'[{AR}]',w) and nz(w)!=w.replace('ة','ه'): pass
                if re.search('[{}]'.format(AR),c): fallback[w]+=1
        # rebuild lines
        k=0; rl=[]
        for ln in lines:
            m=len(ln.split()); rl.append(' '.join(res[k:k+m])); k+=m
        raw='\n'.join(rl)
        for x,y in zip(rl,lines): assert nz(x)==y.strip(), (r.doc_key,r.article_no,x,y)
        out.append(raw)
    n['text_raw']=out
    pd.DataFrame(DICT_LOG,columns=['doc','art','text_word','raw_word','candidates']).to_csv(os.path.join(logdir,'سجل_3_text_raw.csv'),index=False,encoding='utf-8-sig')
    print('  text_raw:', dict(stats))
    return n


# ---------- الخطوة 4 ----------
def step4_text_raw_spelling(d):
    """تصحيحات إملاء text_raw وحده، واختبار القبول (من step4b.py)."""
    n=d
    RAWFIX={'بالآتى':'بالآتي','يراعي':'يراعى','الادارة':'الإدارة','بإسمه':'باسمه','الاتية':'الآتية','عضوآ':'عضوا','ان':'أن',
     'اتعابه':'أتعابه','تنشا':'تنشأ','بييي':'بيئي','الزامية':'إلزامية','انظمة':'أنظمة','ادارة':'إدارة','بالاقتراع':'بالإقتراع',
     'الاقليمية':'الإقليمية','اعداد':'إعداد','الاشراف':'الإشراف','الية':'آلية','اسلوب':'أسلوب','اعضاء':'أعضاء','أدبيآ':'أدبيا',
     'واعضاء':'وأعضاء','رآيه':'رأيه','اعضائها':'أعضائها','يومآ':'يوما','ومكافاته':'ومكافآته','آدائه':'أدائه','تبدا':'تبدأ',
     'المكافات':'المكافآت','وفقآ':'وفقا','المكافاة':'المكافأة','آداء':'أداء','اهداف':'أهداف','واغراضهما':'وأغراضهما','واصحاب':'وأصحاب',
     'ومسؤغة':'ومسوغة','اشخاص':'أشخاص'}
    cnt={}
    def fix(t):
        for a,b in RAWFIX.items():
            pat=re.compile(rf'(?<![{AR}]){a}(?![{AR}])'); k=len(pat.findall(t))
            if k: cnt[a]=cnt.get(a,0)+k; t=pat.sub(b,t)
        return t
    n['text_raw']=n.text_raw.map(fix)
    missing=[a for a in RAWFIX if a not in cnt]; assert not missing, missing
    for r in n.itertuples():
        for x,y in zip(r.text_raw.split('\n'),r.text.split('\n')): assert nz(x)==y.strip(),(r.doc_key,r.article_no,x)
        assert len(r.text_raw.split('\n'))==len(r.text.split('\n'))
    print('  اختبار القبول: nz(text_raw) == text في كل الأسطر')
    return n


# ---------- الخطوة 5 ----------
def step5_topics(d, CLS, PROJ, TOPX, LAWX=None, LAWKEY=None):
    """الحوكمة م42، أعمدة الموضوعات بكود زميلة التنظيف، المواضيع «الواضحة»، وقرارات المختص القانوني (من step5_topics.py)."""
    sys.path.insert(0, PROJ)
    spec = importlib.util.spec_from_file_location('ltc', CLS); ltc = importlib.util.module_from_spec(spec); spec.loader.exec_module(ltc)
    d = d.copy()
    # 1) الحوكمة م42
    m = (d.doc_key == 'gov') & (d.article_no == 42); i = d.index[m][0]
    for c in ['text', 'text_raw']:
        L = d.at[i, c].split('\n'); assert L[1].endswith(' 1 1'), (c, L[1]); L[1] = L[1][:-4]; d.at[i, c] = '\n'.join(L)
    # 2) أعمدة الموضوعات
    d = d[[c for c in d.columns if c not in ('topic_hits','topics','primary_topic','tfidf_topic','tfidf_score','agrees_with_rules','topic_override')]]
    d = ltc.tfidf_topic_check(ltc.classify_articles(d))
    # 3) المواضيع «الواضحة»
    ws = openpyxl.load_workbook(TOPX, data_only=True)['المواد']
    DK = {'اللائحة': 'reg', 'الحوكمة': 'gov'}
    clear = [{'doc_key': DK[r[0]], 'article_no': int(r[1]), 'topic': r[5], 'reason': 'مراجعة المواضيع 2026-10-25: ' + r[6]}
             for r in ws.iter_rows(min_row=2, values_only=True) if r[7] == 'واضح']
    law = []
    # استثناء موثق: قرار فارغ لكن الموضوع مكتوب، ويُطبّق بقرار فجر (لا يُطبّق غيره من القرارات الفارغة)
    BLANK_DECISION_ACCEPTED = {('reg', 54): 'قرار فجر 2026-10-28: القرار فارغ والموضوع مكتوب، وهو من القائمة ويطابق ما اختاره المختص لتوأمها م15'}
    if LAWX:
        import csv
        wb = openpyxl.load_workbook(LAWX, data_only=True)
        g = wb['المطلوب']; who = {g.cell(r, 1).value: g.cell(r, 2).value for r in range(1, g.max_row + 1)}
        name, role, date = who.get('الاسم'), who.get('الصفة / التخصص'), who.get('تاريخ المراجعة')
        if hasattr(date, 'strftime'): date = date.strftime('%Y-%m-%d')
        key = {int(r['م']): r for r in csv.DictReader(open(LAWKEY, encoding='utf-8-sig'))}
        DN = {'اللائحة التنفيذية': 'reg', 'قواعد الحوكمة': 'gov'}
        cur = {(r.doc_key, r.article_no): r.primary_topic for r in d.itertuples()}
        chosen = {}; errs = []; new = set(); skipped = set()
        for r in wb['المواد'].iter_rows(min_row=2, values_only=True):
            if r[0] is None: continue
            i, doc, no, dec, other, rem = int(r[0]), r[1], int(r[2]), (r[7] or '').strip(), (r[8] or '').strip(), (r[10] or '').strip()
            k = key[i]; kk = (k['doc_key'], int(k['article_no']))
            if kk != (DN[doc], no): errs.append(f'الصف {i}: لا يطابق المفتاح')
            if dec == '' and other and kk in BLANK_DECISION_ACCEPTED:
                dec = 'غيرهما'; print(f'تنبيه: طُبّق رغم القرار الفارغ (استثناء موثق): {doc} م{no} «{other}»')
            if dec == '':
                skipped.add(kk); print(f'تنبيه: لم يُطبّق — القرار فارغ: {doc} م{no}' + (f' (مكتوب في عمود الموضوع: «{other}»)' if other else '')); continue
            if dec == 'أ': t = k['الخيار_أ']
            elif dec == 'ب': t = k['الخيار_ب']
            elif dec == 'غيرهما' and other:
                t = other
                if t not in ltc.TOPICS: new.add((DN[doc], no)); print(f'تنبيه: موضوع جديد ليس من مواضيع التصنيف الـ{len(ltc.TOPICS)}: {doc} م{no} «{t}»')
            else: errs.append(f'{doc} م{no}: القرار ناقص ({dec!r}، {other!r})'); continue
            chosen[kk] = (t, rem)
        if len(key) != len(chosen) + len(skipped) and not errs: errs.append('عدد القرارات لا يساوي عدد المواد في المفتاح')
        fin = lambda kk: chosen[kk][0] if kk in chosen else cur[kk]
        if fin(('reg', 15)) != fin(('reg', 54)):
            errs.append(f'التوأمان اللائحة م15 وم54 بموضوعين مختلفين بعد التطبيق: {fin(("reg", 15))} / {fin(("reg", 54))}')
        if errs: sys.exit('ملف القانوني لم يُطبّق:\n' + '\n'.join(errs))
        for (dk, no), (t, rem) in sorted(chosen.items()):
            if t == cur[(dk, no)]: continue
            info = [x for x in (name, role, date) if x]
            reason = 'رأي المختص القانوني' + (f' ({"، ".join(map(str, info))})' if info else '') + (f': {rem}' if rem else '')
            if (dk, no) in BLANK_DECISION_ACCEPTED: reason += ' [' + BLANK_DECISION_ACCEPTED[(dk, no)] + ']'
            if (dk, no) in new: reason += f' [تنبيه: موضوع جديد، ليس من مواضيع التصنيف الـ{len(ltc.TOPICS)}]'
            law.append({'doc_key': dk, 'article_no': no, 'topic': t, 'reason': reason})
        print('lawyer decisions:', len(chosen), 'changed:', len(law), 'new topics:', len(new), 'skipped:', len(skipped))
    # توأم يأخذ موضوع توأمه بعد قرار القانوني (قرار فجر 2026-10-28)
    if law:
        twin = [{'doc_key': 'reg', 'article_no': 111, 'topic': next(o['topic'] for o in law if (o['doc_key'], o['article_no']) == ('reg', 86)),
                 'reason': 'قرار فجر 2026-10-28: توأم اللائحة م86، يأخذ موضوعها بعد رأي المختص القانوني'}]
        print('twin aligned: reg 111 ->', twin[0]['topic']); law = law + twin
    overlap = {(o['doc_key'], o['article_no']) for o in law} & {(o['doc_key'], o['article_no']) for o in clear + ltc.TOPIC_OVERRIDES}
    assert not overlap, overlap
    ltc.TOPIC_OVERRIDES = ltc.TOPIC_OVERRIDES + clear + law
    d = ltc.apply_overrides(d)
    print('  تعديلات الموضوع:', len(ltc.TOPIC_OVERRIDES))
    return d


def main(argv):
    here = os.path.dirname(os.path.abspath(__file__))
    if not argv:
        IN = os.path.join(here, 'المدخلات')
        argv = [os.path.join(IN, 'legal_articles_full_updated.csv'), os.path.join(IN, 'legal_articles_errors_additions.xlsx'),
                os.path.join(here, 'من_مشروع_زميلة1', 'legal topic classification.py'), os.path.join(here, 'من_مشروع_زميلة1'),
                os.path.join(IN, 'مراجعة_المواضيع.xlsx'), os.path.join(here, 'ناتج_التشغيل')]
        LAWF = os.path.join(IN, 'مراجعة_القانوني_المواضيع.xlsx')
        if os.path.exists(LAWF): argv += [LAWF, os.path.join(IN, 'مفتاح_مراجعة_القانوني_لا_يُرسل.csv')]
    UPD, ADDX, CLS, PROJ, TOPX, OUTDIR = [os.path.abspath(x) for x in argv[:6]]
    LAW = [os.path.abspath(x) for x in argv[6:8]]
    if len(LAW) == 1: sys.exit('ملف القانوني يحتاج مفتاحه معه')
    os.makedirs(OUTDIR, exist_ok=True)
    print('قرارات القانوني:', 'مُضمّنة' if LAW else 'غير مُضمّنة')
    print('==> 1 الإضافات'); d = step1_additions(UPD, ADDX, OUTDIR)
    print('==> 2 التصحيحات وأرقام البنود'); d = step2_fixes_and_markers(d, OUTDIR)
    print('==> 3 إعادة بناء text_raw'); d = step3_rebuild_text_raw(d, ADDX, OUTDIR)
    print('==> 4 إملاء text_raw واختبار القبول'); d = step4_text_raw_spelling(d)
    print('==> 5 الموضوعات'); d = step5_topics(d, CLS, PROJ, TOPX, *(LAW or [None, None]))
    out = os.path.join(OUTDIR, 'legal_articles_full.csv')
    d.to_csv(out, index=False, encoding='utf-8-sig')
    print('الناتج:', out)



## التشغيل

In [ ]:
__file__ = '/content/clean_legal_articles.py'   # الكود يحدد مجلده من __file__ (غير معرّف في الدفاتر)
OUT = '/content/ناتج_التنظيف'
main([got['legal_articles_full_updated.csv'], got['legal_articles_errors_additions.xlsx'], got['legal topic classification.py'], W,
      got['مراجعة_المواضيع.xlsx'], OUT] + LAW)

## تنزيل الناتج

In [ ]:
files.download(f'{OUT}/legal_articles_full.csv')